In [ ]:
from pathlib import Path
from ai_news_digest.config import Settings
from ai_news_digest.pipeline import collect_articles, rank_and_select, summarize_and_build, _get_llm
from datetime import datetime

In [ ]:
# Parámetros de corrida
DAYS = 30
TOP_N = 20 ##15
#ENSURE_SOURCES = [] 
ENSURE_SOURCES = ['Kdnuggets.com']
MONTH_NAME = "marzo_26"
LANG = "es"

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_PATH = "outputs/Highlights_AI" + "_"+ MONTH_NAME + "_" + ts +".txt"
print(f"Output path: {OUT_PATH}")

settings = Settings()

In [ ]:
# Cliente LLM compartido — acumula tokens entre ranking y summary
llm_client = _get_llm(settings)
print(f"LLM client: {type(llm_client).__name__ if llm_client else 'None'}")

## Articles Collect

In [ ]:
print("🔎 [1/4] Recolectando artículos...")
df_all = collect_articles(settings, days=DAYS)
print(f"   → Artículos recolectados: {len(df_all)}")
if df_all.empty:
        print("   ⚠️  No se encontraron artículos.")

df_all.head(5)

In [ ]:
df_all.fuente.value_counts()

In [ ]:
df_all.to_csv("outputs/all_articles_mar_26_v2.csv", index=False)

## Articles Ranking

In [ ]:
import pandas as pd
df_all = pd.read_csv("outputs/all_articles_mar_26.csv")
df_all.fuente.value_counts()
df_all.head(5)

In [ ]:
print("📊 [2/4] Rankeando artículos por relevancia...")
df_top = rank_and_select(df_all, settings, top_n=TOP_N, ensure_source=ENSURE_SOURCES, llm_client=llm_client)
print(f"   → Seleccionados top {len(df_top)} artículos")
if llm_client and hasattr(llm_client, "token_summary"):
    print("   ", llm_client.token_summary())

In [ ]:
df_top.fuente.value_counts()

In [ ]:
df_top.to_csv("outputs/top_ranked_articles_mar_26_openAI.csv", index=False)

## Article Sumarize

In [ ]:
# import pandas as pd
# df_top = pd.read_csv("outputs/top_ranked_articles_mar_26.csv")
# print(df_top.fuente.value_counts())
# df_top.head(5)

In [ ]:
print("📝 [3/4] Generando resúmenes y armando artículo...")
article, df_res = summarize_and_build(df_top, settings, month_name=MONTH_NAME, lang=LANG, llm_client=llm_client)
print("   → Resúmenes generados")
if llm_client and hasattr(llm_client, "token_summary"):
    print("   ", llm_client.token_summary())

In [ ]:
df_res.resumen

In [ ]:
article

In [ ]:
print("💾 [4/4] Guardando resultados...")
Path(OUT_PATH).write_text(article, encoding="utf-8")
csv_path = Path(OUT_PATH).with_suffix(".csv")
csv_path.write_text(df_res.to_csv(index=False), encoding="utf-8")
print(f"   → Guardado {OUT_PATH} y {csv_path.name}")

print("✅ Pipeline finalizado con éxito.")

In [ ]:
# Total de tokens usados en todo el pipeline
if llm_client and hasattr(llm_client, "token_summary"):
    print("📊 Token usage total del pipeline:")
    print("  ", llm_client.token_summary())
else:
    print("No LLM client disponible — no hay tokens que reportar.")